### 1. Objetivo y alcance

##### Target Analysis

Definir los objetivos de regresión y clasificación, y la unidad
de análisis para clustering.

Exploración inicial: agosto–septiembre de 2026.
Las decisiones se revisarán al ampliar el periodo.

In [ ]:
from mibici.pipeline import load_stage
import pandas as pd
data = load_stage("quality_fixes")

trips = data["viajes"].copy()
stations = data["estaciones"].copy()

2026-10-17 14:50:01.278 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/mibici/mibici_predictions


### P1 — Regression

- Unidad: estación y día.
- Objetivo: cantidad de salidas registradas.
- Horizonte propuesto: predecir el día siguiente.
- Pendiente: decidir el tratamiento de viajes con duración cero.

In [7]:
daily_departures = (
    trips.assign(date=trips["Inicio_del_viaje"].dt.normalize())
    .groupby(["Origen_Id", "date"])
    .size()
    .rename("departures")
    .reset_index()
)

display(daily_departures.head())
display(daily_departures["departures"].describe())

,Origen_Id,date,departures
0,2,2026-06-01,67
1,2,2026-06-02,46
2,2,2026-06-03,58
3,2,2026-06-04,47
4,2,2026-06-05,40


count    53303.000000
mean        30.512016
std         35.161910
min          1.000000
25%          9.000000
50%         20.000000
75%         40.000000
max        520.000000
Name: departures, dtype: float64

In [8]:
#continuidad temporal de cada estación

print(f"Primera fecha: {daily_departures['date'].min()}")
print(f"Última fecha: {daily_departures['date'].max()}")

station_coverage = (
    daily_departures.groupby("Origen_Id")
    .agg(
        days_with_departures=("date", "nunique"),
        total_departures=("departures", "sum"),
    )
)

display(station_coverage["days_with_departures"].describe())
display(station_coverage.sort_values("days_with_departures").head(10))

Primera fecha: 2026-06-01 00:00:00
Última fecha: 2026-09-30 00:00:00


count    471.000000
mean     113.169851
std       19.066536
min        1.000000
25%      117.000000
50%      122.000000
75%      122.000000
max      122.000000
Name: days_with_departures, dtype: float64

,days_with_departures,total_departures
Origen_Id,,
405,1,3
180,20,493
407,39,1957
440,45,1235
519,47,291
518,49,568
517,50,387
446,50,559
514,51,159


In [9]:
station_coverage = (
    daily_departures.groupby("Origen_Id")
    .agg(
        first_departure_date=("date", "min"),
        last_departure_date=("date", "max"),
        days_with_departures=("date", "nunique"),
        total_departures=("departures", "sum"),
    )
)

display(
    station_coverage
    .sort_values("days_with_departures")
    .head(10)
)

,first_departure_date,last_departure_date,days_with_departures,total_departures
Origen_Id,,,,
405,2026-06-08,2026-06-08,1,3
180,2026-09-11,2026-09-30,20,493
407,2026-06-11,2026-07-21,39,1957
440,2026-08-11,2026-09-30,45,1235
519,2026-08-14,2026-09-30,47,291
518,2026-08-13,2026-09-30,49,568
517,2026-08-12,2026-09-30,50,387
446,2026-08-12,2026-09-30,50,559
514,2026-08-10,2026-09-30,51,159


In [10]:
station_coverage["days_in_span"] = (
    station_coverage["last_departure_date"]
    - station_coverage["first_departure_date"]
).dt.days + 1

station_coverage["days_without_departures_in_span"] = (
    station_coverage["days_in_span"]
    - station_coverage["days_with_departures"]
)

display(
    station_coverage
    .sort_values("days_with_departures")
    .head(10)
)

,first_departure_date,last_departure_date,days_with_departures,total_departures,days_in_span,days_without_departures_in_span
Origen_Id,,,,,,
405,2026-06-08,2026-06-08,1,3,1,0
180,2026-09-11,2026-09-30,20,493,20,0
407,2026-06-11,2026-07-21,39,1957,41,2
440,2026-08-11,2026-09-30,45,1235,51,6
519,2026-08-14,2026-09-30,47,291,48,1
518,2026-08-13,2026-09-30,49,568,49,0
517,2026-08-12,2026-09-30,50,387,50,0
446,2026-08-12,2026-09-30,50,559,50,0
514,2026-08-10,2026-09-30,51,159,52,1


In [12]:
station_coverage = (
    daily_departures.groupby("Origen_Id")
    .agg(
        first_departure_date=("date", "min"),
        last_departure_date=("date", "max"),
        days_with_departures=("date", "nunique"),
        total_departures=("departures", "sum"),
    )
)

station_coverage["days_in_span"] = (
    station_coverage["last_departure_date"]
    - station_coverage["first_departure_date"]
).dt.days + 1

station_coverage["days_without_departures_in_span"] = (
    station_coverage["days_in_span"]
    - station_coverage["days_with_departures"]
)

station_coverage["station_name"] = station_coverage.index.map(
    stations.set_index("id")["name"]
)


print(f"Desde: {daily_departures['date'].min()}")
print(f"Hasta: {daily_departures['date'].max()}")

display(
    station_coverage
    .sort_values("days_with_departures")
    .head(15)
)


Desde: 2026-06-01 00:00:00
Hasta: 2026-09-30 00:00:00


,first_departure_date,last_departure_date,days_with_departures,total_departures,days_in_span,days_without_departures_in_span,station_name
Origen_Id,,,,,,,
405,2026-06-08,2026-06-08,1,3,1,0,<NA>
180,2026-09-11,2026-09-30,20,493,20,0,(GDL-113) C. Ghilardi / C. Arista
407,2026-06-11,2026-07-21,39,1957,41,2,<NA>
440,2026-08-11,2026-09-30,45,1235,51,6,(GDL-301) C. José María Vigil / Av. Américas
519,2026-08-14,2026-09-30,47,291,48,1,(ZPN-107) C. Colina Aria / Av. Díaz de León
518,2026-08-13,2026-09-30,49,568,49,0,(ZPN-108) Av. Atemajac / Av. Yermo y Parra
517,2026-08-12,2026-09-30,50,387,50,0,(ZPN-106) Av. Atemajac/Colinas del Capitolio
446,2026-08-12,2026-09-30,50,559,50,0,(GDL-305) C. Huicholes / Av. México
514,2026-08-10,2026-09-30,51,159,52,1,(ZPN-101) C. Volcán Quinceo/C. Ignacio C.


In [14]:
daily_arrivals = (
    trips.assign(date=trips["Fin_del_viaje"].dt.normalize())
    .groupby(["Destino_Id", "date"])
    .size()
    .rename("arrivals")
    .reset_index()
    .rename(columns={"Destino_Id": "station_id"})
)

In [ ]:

selected_stations = (
    station_coverage
    .sort_values("days_with_departures")
    .head(10)
    .index
)

calendar = pd.MultiIndex.from_product(
    [
        selected_stations,
        pd.date_range("2026-06-01", "2026-09-30", freq="D"),
    ],
    names=["station_id", "date"],
)

station_daily_activity = (
    daily_departures
    .rename(columns={"Origen_Id": "station_id"})
    .set_index(["station_id", "date"])[["departures"]]
    .join(
        daily_arrivals.set_index(["station_id", "date"]),
        how="outer",
    )
    .reindex(calendar)
    .fillna(0)
    .astype("int64")
    .reset_index()
)

In [18]:
active_days = station_daily_activity.loc[
    station_daily_activity["departures"].gt(0)
    | station_daily_activity["arrivals"].gt(0)
]

activity_span = active_days.groupby("station_id").agg(
    first_activity=("date", "min"),
    last_activity=("date", "max"),
    days_with_activity=("date", "nunique"),
)

activity_span["days_in_span"] = (
    activity_span["last_activity"]
    - activity_span["first_activity"]
).dt.days + 1

activity_span["no_activity_inside_span"] = (
    activity_span["days_in_span"]
    - activity_span["days_with_activity"]
)

display(activity_span)

,first_activity,last_activity,days_with_activity,days_in_span,no_activity_inside_span
station_id,,,,,
180,2026-09-11,2026-09-30,20,20,0
405,2026-06-08,2026-06-08,1,1,0
407,2026-06-11,2026-07-21,39,41,2
440,2026-08-10,2026-09-30,49,52,3
446,2026-08-12,2026-09-30,50,50,0
514,2026-08-10,2026-09-30,52,52,0
516,2026-08-11,2026-09-30,51,51,0
517,2026-08-12,2026-09-30,50,50,0
518,2026-08-12,2026-09-30,50,50,0


In [19]:
activity_check = station_daily_activity.merge(
    activity_span[["first_activity", "last_activity"]],
    left_on="station_id",
    right_index=True,
    how="left",
)

internal_gaps = activity_check.loc[
    activity_check["date"].between(
        activity_check["first_activity"],
        activity_check["last_activity"],
    )
    & activity_check["departures"].eq(0)
    & activity_check["arrivals"].eq(0),
    ["station_id", "date"],
]

display(internal_gaps)

,station_id,date
278,407,2026-07-05
279,407,2026-07-06
438,440,2026-08-12
440,440,2026-08-14
441,440,2026-08-15


In [20]:
network_departures = (
    trips.groupby(trips["Inicio_del_viaje"].dt.normalize())
    .size()
    .rename("network_departures")
)

gap_review = internal_gaps.copy()
gap_review["network_departures"] = (
    gap_review["date"].map(network_departures).fillna(0).astype("int64")
)

display(gap_review)

,station_id,date,network_departures
278,407,2026-07-05,9930
279,407,2026-07-06,13483
438,440,2026-08-12,14280
440,440,2026-08-14,14296
441,440,2026-08-15,10078


In [21]:
#Ampliamos la seleccion a analizar
station_ids = pd.Index(
    pd.concat([
        trips["Origen_Id"],
        trips["Destino_Id"],
    ]).unique(),
    name="station_id",
)

calendar = pd.MultiIndex.from_product(
    [
        station_ids,
        pd.date_range("2026-06-01", "2026-09-30", freq="D"),
    ],
    names=["station_id", "date"],
)

station_daily_activity = (
    daily_departures
    .rename(columns={"Origen_Id": "station_id"})
    .set_index(["station_id", "date"])[["departures"]]
    .join(
        daily_arrivals.set_index(["station_id", "date"]),
        how="outer",
    )
    .reindex(calendar)
    .fillna(0)
    .astype("int64")
    .reset_index()
)

In [22]:
has_activity = (
    station_daily_activity["departures"].gt(0)
    | station_daily_activity["arrivals"].gt(0)
)

activity_span = (
    station_daily_activity.loc[has_activity]
    .groupby("station_id")
    .agg(
        first_activity=("date", "min"),
        last_activity=("date", "max"),
        days_with_activity=("date", "nunique"),
    )
)

activity_span["days_in_span"] = (
    activity_span["last_activity"]
    - activity_span["first_activity"]
).dt.days + 1

activity_span["no_activity_inside_span"] = (
    activity_span["days_in_span"]
    - activity_span["days_with_activity"]
)

print(f"Estaciones revisadas: {len(activity_span):,}")
print(
    "Estaciones con huecos internos:",
    activity_span["no_activity_inside_span"].gt(0).sum(),
)

display(
    activity_span
    .sort_values("no_activity_inside_span", ascending=False)
    .head(15)
)

Estaciones revisadas: 471
Estaciones con huecos internos: 103


,first_activity,last_activity,days_with_activity,days_in_span,no_activity_inside_span
station_id,,,,,
15,2026-06-01,2026-09-30,59,122,63
35,2026-06-01,2026-09-30,82,122,40
497,2026-07-02,2026-09-30,77,91,14
244,2026-06-01,2026-09-30,110,122,12
42,2026-06-01,2026-09-30,113,122,9
4,2026-06-01,2026-09-30,115,122,7
346,2026-06-01,2026-09-30,116,122,6
290,2026-06-01,2026-09-30,116,122,6
303,2026-06-01,2026-09-30,116,122,6


In [23]:
activity_check = station_daily_activity.merge(
    activity_span[["first_activity", "last_activity"]],
    left_on="station_id",
    right_index=True,
    how="left",
)

internal_gaps = activity_check.loc[
    activity_check["date"].between(
        activity_check["first_activity"],
        activity_check["last_activity"],
    )
    & activity_check["departures"].eq(0)
    & activity_check["arrivals"].eq(0)
]

gap_dates = (
    internal_gaps.groupby("date")["station_id"]
    .nunique()
    .rename("stations_without_activity")
    .to_frame()
)

gap_dates["network_departures"] = (
    gap_dates.index.to_series()
    .map(network_departures)
    .fillna(0)
    .astype("int64")
)

display(
    gap_dates.sort_values(
        "stations_without_activity",
        ascending=False,
    ).head(15)
)

,stations_without_activity,network_departures
date,,
2026-06-05,9,9538
2026-06-07,9,8418
2026-06-19,9,13135
2026-07-26,9,8771
2026-06-06,8,10132
2026-06-28,8,8352
2026-07-01,8,11600
2026-06-04,8,11912
2026-06-18,7,13434


### P2 — Classification

No contamos con inventario observado de bicicletas ni anclajes libres.

Se debe definir un proxy de riesgo basado en flujos de viajes y
capacidad. No se interpretará como disponibilidad real.

Pendiente: definir fórmula, umbral, horizonte y momento de predicción.
Revisar las estaciones con capacidad cero antes de construir el proxy.

### P3 — Clustering

- Unidad: estación.
- No existe una variable objetivo supervisada.
- Se explorarán perfiles horarios, diferencias entre días laborales
  y fines de semana, y ubicación.
- Pendiente: definir periodo, estaciones elegibles y variables.